# Development notebook: Indian ANPR data, OCR evaluation and detector data

This notebook is the working log for the ANPR project. It is kept as it was run, **including failed attempts**, which are marked as such in the explanation above each cell. The final, cleaned code lives in the GitHub repo (`sushovanjena/ANPR`); the notebook shows how it was built and what was learned.

**Parts**
1. Repo scaffold, lint/test setup and pushing to GitHub
2. Kaggle plate dataset: download and inventory
3. DataCluster sample dataset
4. RoadText-1K: building validation and test crops of real plates from video
5. OCR baseline: pretrained SVTRv2 vs a vision-language teacher on those crops
6. Detector tooling and data: detection metrics, COCO backgrounds, composite small-plate data, final training splits

Runtime notes: Colab runtimes were recycled several times, so many cells start by restoring code and data from Drive (`MyDrive/bosch_anpr`). Secrets (GitHub, Kaggle) come from Colab Secrets and are never printed in full.

## Part 1: Repo scaffold, lint and tests, GitHub

Creates the project scaffold inside the cloned repo: `pyproject.toml` (build settings, dependencies, ruff and pytest configuration), the `anpr` package with one-line docstrings, a seeding utility `utils/seed.py` with its first test, a `.gitignore`, and empty `configs/`, `docs/` and `notebooks/` folders.

In [ ]:
# Cell 3b: create the project scaffold inside the cloned repo
from pathlib import Path

REPO_DIR = Path("/content/ANPR")

files = {
"pyproject.toml": '''[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "anpr"
version = "0.1.0"
description = "ANPR for Indian traffic scenes, targeting NVIDIA Jetson Orin"
requires-python = ">=3.10"
dependencies = [
  "ultralytics", "opencv-python-headless", "albumentations",
  "numpy", "pandas", "pyyaml", "matplotlib", "seaborn",
  "onnx", "onnxruntime-gpu", "imagehash", "tqdm",
]

[project.optional-dependencies]
dev = ["pytest", "ruff", "black"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.ruff]
line-length = 88
src = ["src"]

[tool.ruff.lint]
select = ["E", "F", "W", "D", "I"]
ignore = ["D203", "D213"]

[tool.ruff.lint.per-file-ignores]
"tests/*" = ["D"]

[tool.pytest.ini_options]
testpaths = ["tests"]
''',
"src/anpr/__init__.py": '"""ANPR for Indian traffic scenes."""\n',
"src/anpr/data/__init__.py": '"""Dataset loading, conversion and validation."""\n',
"src/anpr/models/__init__.py": '"""Detector and recognizer wrappers."""\n',
"src/anpr/eval/__init__.py": '"""Metrics and error analysis."""\n',
"src/anpr/deploy/__init__.py": '"""Export and edge-runtime helpers."""\n',
"src/anpr/utils/__init__.py": '"""Shared utilities."""\n',
"src/anpr/utils/seed.py": '''"""Reproducibility helpers."""
import os
import random

import numpy as np


def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy and (if installed) PyTorch RNGs.

    Args:
        seed: Value used to seed every random number generator.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
''',
"tests/test_seed.py": '''import numpy as np

from anpr.utils.seed import seed_everything


def test_seed_is_reproducible():
    seed_everything(0)
    first = np.random.rand(3)
    seed_everything(0)
    assert np.allclose(first, np.random.rand(3))
''',
".gitignore": "__pycache__/\n*.egg-info/\n.ipynb_checkpoints/\ndata/\nruns/\nexports/\n*.pt\n*.onnx\n*.engine\n",
}

for rel, content in files.items():
    path = REPO_DIR / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)

for d in ["configs", "docs", "notebooks"]:
    (REPO_DIR / d).mkdir(exist_ok=True)
    (REPO_DIR / d / ".gitkeep").touch()

Installs the package in editable mode, then runs ruff (lint) and pytest. The lint step reports one docstring-format error (rule D413); the next cell fixes it.

In [ ]:
%cd /content/ANPR
!pip install -q -e ".[dev]"
!ruff check src tests && pytest -q

Declares the Google docstring convention for ruff's pydocstyle rules in `pyproject.toml`, which resolves the D413 complaint.

In [ ]:
# Cell 3c: declare the Google docstring convention in pyproject.toml
from pathlib import Path

p = Path("/content/ANPR/pyproject.toml")
text = p.read_text()
text = text.replace(
    '[tool.ruff.lint.per-file-ignores]',
    '[tool.ruff.lint.pydocstyle]\nconvention = "google"\n\n[tool.ruff.lint.per-file-ignores]',
)
p.write_text(text)

Re-runs ruff and pytest after the config fix: all checks pass and the one seed test passes.

In [ ]:
%cd /content/ANPR
!ruff check src tests && pytest -q

Sanity check that the repo folder exists and lists its contents (`src`, `tests`, `configs`, `docs`, `notebooks`).

In [ ]:
from pathlib import Path
REPO_DIR = Path("/content/ANPR")
print("repo exists:", REPO_DIR.exists())
!ls -a {REPO_DIR}

**Failed attempt.** First try to commit and push the scaffold to GitHub using a token from Colab Secrets. It stops with `fatal: not a git repository`: the folder was created locally and had no git history. The following cells set this up properly.

In [ ]:
%cd /content/ANPR
from google.colab import userdata
token = userdata.get("GH_TOKEN")

!git config user.name "sushovanjena"
!git config user.email "sushovanjena@gmail.com"
!git add -A
!git commit -m "Add project scaffold, seed util and first test"
!git push https://{token}@github.com/sushovanjena/ANPR.git HEAD

Adds the ruff and pytest caches to `.gitignore`, sets the git identity and stages the files, then prints the staged files so the commit contents can be checked before pushing.

In [ ]:
!printf ".ruff_cache/\n.pytest_cache/\n" >> .gitignore
!git config user.name "sushovanjena"
!git config user.email "sushovanjena@gmail.com"
!git add -A
!git status --short

Commits the scaffold ('Add project scaffold, seed util and first test') and pushes it, creating the `main` branch of `sushovanjena/ANPR`.

In [ ]:
!git commit -m "Add project scaffold, seed util and first test"
!git push https://{token}@github.com/sushovanjena/ANPR.git HEAD:main

Confirms the pydocstyle setting is present in `pyproject.toml`.

In [ ]:
!grep -A1 pydocstyle pyproject.toml

Rewrites `.gitignore` so the data, runs and exports rules are anchored to the repository root (`/data/` and so on, so they do not hide `src/anpr/data`), creates the `anpr.data` package and stages the changes.

In [ ]:
%cd /content/ANPR
from pathlib import Path

gi = Path(".gitignore")
gi.write_text(
    "__pycache__/\n*.egg-info/\n.ipynb_checkpoints/\n"
    ".ruff_cache/\n.pytest_cache/\n"
    "/data/\n/runs/\n/exports/\n"
    "*.pt\n*.onnx\n*.engine\n"
)

Path("src/anpr/data").mkdir(parents=True, exist_ok=True)
Path("src/anpr/data/__init__.py").write_text('"""Dataset loading, conversion and validation."""\n')

!git add -A
!git status --short

Reads the GitHub token from Colab Secrets (whitespace stripped), commits 'Anchor data ignore rules to repo root; add data package' and pushes to `main`.

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN").strip()

!git commit -m "Anchor data ignore rules to repo root; add data package"
!git push https://{token}@github.com/sushovanjena/ANPR.git HEAD:main

Attaches git history to the existing project folder: clones the repo into a temporary folder, copies its `.git` into `/content/ANPR`, removes the token from the stored remote URL and shows `git status`. Used when a folder has files but no git metadata.

In [ ]:
%cd /content
from google.colab import userdata
token = userdata.get("GH_TOKEN").strip()

# 1. clone into a temp folder, only to get its .git directory
!rm -rf /content/ANPR_tmp
!git clone https://{token}@github.com/sushovanjena/ANPR.git /content/ANPR_tmp

# 2. move .git into your existing project folder
!cp -r /content/ANPR_tmp/.git /content/ANPR/.git
!rm -rf /content/ANPR_tmp

# 3. don't keep the token inside the repo config
%cd /content/ANPR
!git remote set-url origin https://github.com/sushovanjena/ANPR.git
!git branch -M main
!git status --short

Creates a placeholder `inventory.py` and checks that git sees it as a new untracked file; this prepares the dataset-inventory module written in a later cell.

In [ ]:
Path("src/anpr/data/inventory.py").write_text("# placeholder\n")
!git status --short src/anpr/data

**Debug cell for the failing push.** Prints the token length and a short prefix (never the full token) and runs `git push --dry-run` to test authentication. It still reports 'not a git repository', which confirmed the folder was not yet attached to git at that point.

In [ ]:
%cd /content/ANPR
from google.colab import userdata

token = userdata.get("GH_TOKEN").strip()          # .strip() removes stray whitespace/newlines
print("token length:", len(token), "| prefix:", token[:11])   # prefix only; safe to print

!git push --dry-run https://{token}@github.com/sushovanjena/ANPR.git HEAD

A restore routine that handles the three possible states of `/content/ANPR`: already a repo (fast-forward pull), files without git (attach `.git` from a temporary clone), or missing (clone). Ends by listing the latest commits.

In [ ]:
from pathlib import Path
from google.colab import userdata

REPO_DIR = Path("/content/ANPR")
token = userdata.get("GH_TOKEN").strip()
AUTH_URL = f"https://{token}@github.com/sushovanjena/ANPR.git"
PLAIN_URL = "https://github.com/sushovanjena/ANPR.git"

if (REPO_DIR / ".git").exists():
    !git -C {REPO_DIR} pull --ff-only {AUTH_URL} main
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    # files exist but no git metadata: attach .git without touching the files
    !rm -rf /content/ANPR_tmp
    !git clone {AUTH_URL} /content/ANPR_tmp
    !cp -r /content/ANPR_tmp/.git {REPO_DIR}/.git
    !rm -rf /content/ANPR_tmp
    !git -C {REPO_DIR} reset -q
    !git -C {REPO_DIR} ls-files --deleted | xargs -r git -C {REPO_DIR} checkout --
else:
    !git clone {AUTH_URL} {REPO_DIR}

# keep the token out of the repo's saved config
!git -C {REPO_DIR} remote set-url origin {PLAIN_URL}
!git -C {REPO_DIR} branch -M main
!git config --global user.name "sushovanjena"
!git config --global user.email "sushovanjena@gmail.com"

%cd {REPO_DIR}
!pip install -q -e ".[dev]"
!git status --short
!git log --oneline | head -3

Lint and test run. **Fails** ruff because the placeholder `inventory.py` has no module docstring; the next cell replaces it with the real module.

In [ ]:
!ruff check src tests && pytest -q

Writes the real `anpr/data/inventory.py` (lists images, reads YOLO label files, builds a per-box table with integrity flags such as missing label file or malformed lines) and its tests. Lint is clean and 4 tests pass.

In [ ]:
%cd /content/ANPR
from pathlib import Path

Path("src/anpr/data/inventory.py").write_text('''"""Inventory and integrity checks for YOLO-format detection datasets."""
from __future__ import annotations

from pathlib import Path

import pandas as pd
from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(split_dir: Path) -> list[Path]:
    """Return all image files below ``split_dir``, sorted."""
    return sorted(p for p in split_dir.rglob("*") if p.suffix.lower() in IMG_EXTS)


def label_path_for(image_path: Path) -> Path:
    """Map an image path to its YOLO label path.

    Handles both ``.../images/x.jpg -> .../labels/x.txt`` and labels
    stored next to the image.
    """
    parts = list(image_path.parts)
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        return Path(*parts).with_suffix(".txt")
    return image_path.with_suffix(".txt")


def parse_label_file(path: Path) -> tuple[list[tuple], int]:
    """Parse a YOLO label file.

    Returns:
        A tuple (rows, n_malformed). Rows are (cls, cx, cy, w, h) tuples.
        Lines without exactly 5 fields (for example polygons) count as
        malformed.
    """
    if not path.exists():
        return [], 0
    rows, bad = [], 0
    for line in path.read_text().splitlines():
        fields = line.split()
        if len(fields) != 5:
            bad += 1
            continue
        rows.append((int(float(fields[0])), *map(float, fields[1:])))
    return rows, bad


def build_box_table(split: str, image_paths: list[Path]) -> pd.DataFrame:
    """Build a table with one row per box, or one empty row per unlabeled image."""
    records = []
    for img in image_paths:
        with Image.open(img) as im:  # reads the header only
            w_img, h_img = im.size
        lab = label_path_for(img)
        rows, bad = parse_label_file(lab)
        base = {
            "split": split,
            "image": str(img),
            "img_w": w_img,
            "img_h": h_img,
            "label_exists": lab.exists(),
            "n_malformed": bad,
        }
        if not rows:
            records.append({**base, "cls": None})
        for cls, cx, cy, w, h in rows:
            records.append(
                {
                    **base,
                    "cls": cls,
                    "cx": cx,
                    "cy": cy,
                    "w": w,
                    "h": h,
                    "area_rel": w * h,
                    "px_w": w * w_img,
                    "px_h": h * h_img,
                    "aspect": (w * w_img) / max(h * h_img, 1e-9),
                }
            )
    return pd.DataFrame.from_records(records)
''')

Path("tests/test_inventory.py").write_text('''from pathlib import Path

from anpr.data.inventory import label_path_for, parse_label_file


def test_parse_valid_and_malformed(tmp_path):
    f = tmp_path / "a.txt"
    f.write_text("0 0.5 0.5 0.2 0.1\\n0 0.1 0.1 0.2 0.2 0.3 0.3\\n")
    rows, bad = parse_label_file(f)
    assert rows == [(0, 0.5, 0.5, 0.2, 0.1)] and bad == 1


def test_missing_label_is_empty(tmp_path):
    assert parse_label_file(tmp_path / "none.txt") == ([], 0)


def test_label_path_mapping():
    p = Path("train/images/x.jpg")
    assert label_path_for(p) == Path("train/labels/x.txt")
''')

!wc -l src/anpr/data/inventory.py tests/test_inventory.py
!ruff check src tests && pytest -q

Commits 'Add dataset inventory module and tests' and pushes to `main`.

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN").strip()

!git add -A
!git status --short
!git commit -m "Add dataset inventory module and tests"
!git push https://{token}@github.com/sushovanjena/ANPR.git HEAD:main

## Part 2: Kaggle plate dataset

Downloads the Kaggle dataset `kedarsai/indian-license-plates-with-labels` (credentials from Colab Secrets) into `/content/data/raw` and lists its structure: an `images` and a `labels` folder, 4,104 files in total.

In [ ]:
!pip install -q kaggle
import os
from pathlib import Path
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

LOCAL = Path("/content/data"); LOCAL.mkdir(exist_ok=True)
!kaggle datasets download -d kedarsai/indian-license-plates-with-labels -p {LOCAL}/raw --unzip
!find {LOCAL}/raw -maxdepth 3 -type d | head -20
!find {LOCAL}/raw -type f | wc -l

Re-installs the repo in editable mode after a runtime restart, refreshes Python's import caches and checks that `import anpr` resolves to `/content/ANPR/src`.

In [ ]:
%cd /content/ANPR
!pip install -q -e ".[dev]"

import importlib, sys
importlib.invalidate_caches()
import anpr
print("anpr found at:", anpr.__file__)

Confirms the import works and that the raw data folders are in place.

In [ ]:
%cd /content/ANPR
import anpr
print("anpr found at:", anpr.__file__)
!ls /content/data/raw | head

Runs the inventory over the Kaggle data: counts images and boxes, images without a label file, images with zero boxes and malformed label lines, shows how many boxes each image has (most have one, a few have several) and box-size statistics.

In [ ]:
import pandas as pd
from pathlib import Path
from anpr.data.inventory import list_images, build_box_table

LOCAL = Path("/content/data")
RAW = LOCAL / "raw"
df = build_box_table("all", list_images(RAW))
imgs = df.drop_duplicates("image").reset_index(drop=True)
boxes = df.dropna(subset=["cls"])

print("images:", len(imgs), "| boxes:", len(boxes))
print("no label file:", (~imgs.label_exists).sum(), "| zero boxes:", imgs.cls.isna().sum(),
      "| malformed lines:", imgs.n_malformed.sum())
print(boxes.cls.value_counts())
print(imgs[["img_w", "img_h"]].value_counts().head(10))
print(boxes[["px_w", "px_h", "area_rel", "aspect"]].describe(percentiles=[.05, .5, .95]))
print(boxes.groupby("image").size().value_counts().sort_index())

Same restore logic as above (pull, attach `.git` or clone), re-run after a runtime restart; ends by listing the latest commits.

In [ ]:
from pathlib import Path
from google.colab import userdata

REPO_DIR = Path("/content/ANPR")
token = userdata.get("GH_TOKEN").strip()
AUTH_URL = f"https://{token}@github.com/sushovanjena/ANPR.git"
PLAIN_URL = "https://github.com/sushovanjena/ANPR.git"

if (REPO_DIR / ".git").exists():
    !git -C {REPO_DIR} pull --ff-only {AUTH_URL} main
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    !rm -rf /content/ANPR_tmp
    !git clone {AUTH_URL} /content/ANPR_tmp
    !cp -r /content/ANPR_tmp/.git {REPO_DIR}/.git
    !rm -rf /content/ANPR_tmp
    !git -C {REPO_DIR} reset -q
else:
    !git clone {AUTH_URL} {REPO_DIR}

!git -C {REPO_DIR} remote set-url origin {PLAIN_URL}
!git -C {REPO_DIR} branch -M main
!git config --global user.name "sushovanjena"
!git config --global user.email "sushovanjena@gmail.com"

%cd {REPO_DIR}
!pip install -q -e ".[dev]" kaggle
!git log --oneline | head -3


Rebuilds the inventory table and prints image-size statistics (width and height quantiles) for the Kaggle data, downloading it again first if it is missing.

In [ ]:
import sys, os
sys.path.insert(0, "/content/ANPR/src")

import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata
from anpr.data.inventory import list_images, build_box_table

LOCAL = Path("/content/data"); RAW = LOCAL / "raw"
if not (RAW / "images").exists():
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    LOCAL.mkdir(exist_ok=True)
    !kaggle datasets download -d kedarsai/indian-license-plates-with-labels -p {RAW} --unzip

df = build_box_table("all", list_images(RAW))
imgs = df.drop_duplicates("image").reset_index(drop=True)
boxes = df.dropna(subset=["cls"]).copy()

imgs = imgs.merge(boxes.groupby("image").area_rel.max().rename("max_area_rel"),
                  left_on="image", right_index=True, how="left")
imgs["kind"] = np.select(
    [imgs.max_area_rel.isna(), imgs.max_area_rel >= 0.5],
    ["no_label", "crop"], default="scene")

print("images:", len(imgs), "| boxes:", len(boxes))
print(imgs.kind.value_counts())
print("aspect < 1.5:", (boxes.aspect < 1.5).sum(), "| aspect > 8:", (boxes.aspect > 8).sum())
scene_boxes = boxes[boxes.image.isin(imgs[imgs.kind == "scene"].image)]
print(scene_boxes[["px_w", "px_h", "area_rel"]].describe(percentiles=[.05, .5, .95]))
print(imgs[imgs.kind == "scene"][["img_w", "img_h"]].describe())


Defines `show_grid`, which plots a random sample of images with their ground-truth boxes, as a visual check that the labels are correct.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_grid(rows, title, ncols=4):
    rows = rows.sample(min(8, len(rows)), random_state=0)
    fig, axes = plt.subplots(2, ncols, figsize=(12, 6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        im = cv2.cvtColor(cv2.imread(r.image), cv2.COLOR_BGR2RGB)
        for _, b in boxes[boxes.image == r.image].iterrows():
            x1, y1 = int((b.cx - b.w / 2) * r.img_w), int((b.cy - b.h / 2) * r.img_h)
            x2, y2 = int((b.cx + b.w / 2) * r.img_w), int((b.cy + b.h / 2) * r.img_h)
            cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), max(2, r.img_w // 200))
        ax.imshow(im)
        ax.set_title(f"{Path(r.image).name[:16]} {r.img_w}x{r.img_h}", fontsize=8)
    fig.suptitle(title)
    plt.show()

show_grid(imgs[imgs.kind == "scene"], "scene")
show_grid(imgs[imgs.kind == "no_label"], "no label file")
odd = boxes[(boxes.aspect < 1.0) | (boxes.aspect > 8)].image
show_grid(imgs[imgs.image.isin(odd)], "aspect outliers")


Empty cell. It can be deleted.

**Failed attempt.** Tries to restore the project from Drive without git (mount Drive, copy `code_backup/src` to `/content/ANPR`). It stops with `ValueError: mount failed` because Drive had not been authorised on that runtime; later cells mount it successfully.

In [ ]:
import shutil
import sys
from pathlib import Path

import pandas as pd
from google.colab import drive

# Drive-only restore: no git clone, no pip install, no GPU needed for label work
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/bosch_anpr")
REPO = Path("/content/ANPR")
if not (REPO / "src/anpr/ocr").exists():
    REPO.mkdir(exist_ok=True)
    shutil.copytree(DRIVE / "code_backup/src", REPO / "src", dirs_exist_ok=True)
    shutil.copytree(DRIVE / "code_backup/tests", REPO / "tests", dirs_exist_ok=True)
sys.path.insert(0, str(REPO / "src"))

labels = pd.read_csv(DRIVE / "data/processed/ocr_labels_v2.csv", keep_default_na=False)
print(len(labels), "labels loaded |", sorted(p.name for p in (REPO / "src/anpr/ocr").glob("*.py")))


## Part 3: DataCluster sample dataset

Downloads the DataCluster Labs 'Indian number plates' sample dataset from Hugging Face (images plus XML annotations) into `/content/ext/datacluster` and prints one annotation as an example.

In [ ]:
!pip install -q -U huggingface_hub
from pathlib import Path

from huggingface_hub import list_repo_files, snapshot_download

DC = "Dataclusterlabspvtltd/indian-number-plates-dataset"
files = list_repo_files(DC, repo_type="dataset")
print("repo files:", len(files), "| non-image/xml:", [f for f in files if not f.lower().endswith((".jpg", ".jpeg", ".png", ".xml"))][:10])

EXT = Path("/content/ext/datacluster")
snapshot_download(DC, repo_type="dataset", local_dir=str(EXT))
ext_imgs = sorted(p for p in EXT.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
ext_xmls = sorted(EXT.rglob("*.xml"))
print("images:", len(ext_imgs), "| xml files:", len(ext_xmls))
print(ext_xmls[0].name)
print(ext_xmls[0].read_text()[:1500])


Inspects which XML tags the annotations use and reads the dataset README. The README says this is a small sample for evaluation; the full 15,000+ image set is not included.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

tags = Counter()
for p in ext_xmls:
    for el in ET.parse(p).getroot().iter():
        tags[el.tag] += 1
print("tags across all xml:", dict(tags))
first = ET.parse(ext_xmls[0]).getroot()
obj = first.find("object")
print(ET.tostring(obj, encoding="unicode")[:900])
print("README head:")
print((EXT / "README.md").read_text()[:600])


Parses the DataCluster XML into a table of boxes and plate texts and defines `normalize_plate` and the Indian-format patterns (standard and Bharat series) to check which transcribed plates are valid.

In [ ]:
import pandas as pd

import re

STD = re.compile(r"^[A-Z]{2}[0-9]{1,2}[A-Z]{0,3}[0-9]{1,4}$")
BH = re.compile(r"^[0-9]{2}BH[0-9]{4}[A-Z]{1,2}$")


def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", text.upper())


def matches_indian_format(plate):
    return bool(STD.match(plate) or BH.match(plate))


rows = []
for p in ext_xmls:
    root = ET.parse(p).getroot()
    fname = root.findtext("filename")
    w, h = int(root.find("size").findtext("width")), int(root.find("size").findtext("height"))
    for k, obj in enumerate(root.iter("object")):
        bb = obj.find("bndbox")
        attrs = {a.findtext("name"): a.findtext("value") for a in obj.iter("attribute")}
        rows.append({
            "xml": p.name, "filename": fname, "img_w": w, "img_h": h, "k": k,
            "x1": float(bb.findtext("xmin")), "y1": float(bb.findtext("ymin")),
            "x2": float(bb.findtext("xmax")), "y2": float(bb.findtext("ymax")),
            "rotation": attrs.get("rotation"), "text_raw": attrs.get("number_plate_text"),
        })
ext = pd.DataFrame(rows)
ext["text"] = ext.text_raw.fillna("").map(normalize_plate)
ext["has_text"] = ext.text.ne("")
ext["fmt_ok"] = ext.text.map(matches_indian_format)
ext["w_px"] = ext.x2 - ext.x1
ext["h_px"] = ext.y2 - ext.y1
ext["area_rel"] = ext.w_px * ext.h_px / (ext.img_w * ext.img_h)

print("objects:", len(ext), "| images:", ext.filename.nunique(), "| with plate text:", int(ext.has_text.sum()))
print("attribute names seen:", sorted({a.findtext("name") for p in ext_xmls for a in ET.parse(p).getroot().iter("attribute")}))
print("text in standard Indian format:", int(ext.fmt_ok.sum()), "of", int(ext.has_text.sum()))
print("image sizes:", ext[["img_w", "img_h"]].drop_duplicates().head(8).values.tolist())
print(ext[["w_px", "h_px", "area_rel"]].describe(percentiles=[.1, .5, .9]).round(3).to_string())
print(ext[["filename", "k", "rotation", "text_raw", "text"]].to_string(max_colwidth=22))


## Part 4: RoadText-1K (validation and test crops from video)

Checks the RoadText-1K archive (a 14 GB zip) with an HTTP HEAD request. The download is guarded and prints 'not downloading', because the whole file is too large to fetch.

In [ ]:
import collections
import os
import zipfile
from pathlib import Path

import requests

URL = "https://cvit.iiit.ac.in/images/datasets/RoadText-1K/RoadText1k.zip"
h = requests.head(URL, allow_redirects=True, timeout=30)
size = int(h.headers.get("content-length", 0))
print("HEAD:", h.status_code, h.headers.get("content-type"), round(size / 2**20), "MB")

RT = Path("/content/ext/roadtext")
RT.mkdir(parents=True, exist_ok=True)
if h.status_code == 200 and 0 < size < 4 * 2**30:
    !wget -q -O /content/ext/roadtext/RoadText1k.zip {URL}
    with zipfile.ZipFile(RT / "RoadText1k.zip") as z:
        names = z.namelist()
    print(len(names), "entries")
    print(names[:12])
    print(collections.Counter(os.path.splitext(n)[1].lower() for n in names).most_common(8))
else:
    print("not downloading: unexpected size or status")


Opens the remote zip without downloading it (`remotezip`) and lists what is inside: folder layout, file types and the largest members.

In [ ]:
print("accept-ranges:", h.headers.get("accept-ranges"))
!pip install -q remotezip
from remotezip import RemoteZip

rz = RemoteZip(URL)
infos = rz.infolist()
print(len(infos), "entries")
print([i.filename for i in infos[:10]])
print(collections.Counter(os.path.splitext(i.filename)[1].lower() for i in infos).most_common(8))
big = sorted(infos, key=lambda i: -i.file_size)[:3]
print("largest:", [(i.filename, round(i.file_size / 2**20)) for i in big])
print("top-level folders:", collections.Counter(i.filename.split("/")[0] for i in infos).most_common(6))


Reads the licence file and lists the JSON annotation files and videos in the archive (test videos 701 to 1000, average size).

In [ ]:
print(rz.read("RoadText1k/License.txt").decode("utf-8", "replace")[:1500])
print()
json_files = [(i.filename, round(i.file_size / 2**20)) for i in infos if i.filename.endswith(".json")]
print(json_files)
vids = [i for i in infos if i.filename.endswith(".mp4")]
print("videos:", len(vids), "| mean MB:", round(sum(i.file_size for i in vids) / len(vids) / 2**20, 1), "| example names:", [v.filename for v in vids[:3]])


**Failed attempt.** Tries to read a text-transcription JSON straight from the remote zip. The server resets the connection (`ConnectionResetError`) on long streams. The next cell works around this with short ranged downloads.

In [ ]:
import traceback
try:
    import json

    tt = [i for i in infos if "Text_Transcription" in i.filename]
    print([(i.filename.split("/")[-1], i.file_size) for i in tt][:4])
    raw = rz.read(tt[1].filename)
    d = json.loads(raw)
    print(type(d).__name__, "| size:", len(d))
    if isinstance(d, dict):
        ks = list(d.keys())
        print("keys:", ks[:5])
        v = d[ks[0]]
        print(type(v).__name__, json.dumps(v)[:700])
    else:
        print(json.dumps(d[0])[:700])
    loc = [i for i in infos if "Localisation" in i.filename]
    rawl = rz.read(loc[1].filename)
    dl = json.loads(rawl)
    print("LOC", type(dl).__name__, "| size:", len(dl))
    if isinstance(dl, dict):
        kl = list(dl.keys())
        print("keys:", kl[:5])
        print(json.dumps(dl[kl[0]])[:900])
    else:
        print(json.dumps(dl[0])[:900])

except Exception:
    print(traceback.format_exc()[-1500:])


Defines `ranged()`, which fetches byte ranges with a short timeout and retries, plus helpers that extract single members of the remote zip. Used to load the localisation annotations (frames with labelled text boxes).

In [ ]:
import json
import struct
import time
import traceback
import zlib
from pathlib import Path

import requests

RT = Path("/content/ext/roadtext")
RT.mkdir(parents=True, exist_ok=True)


def ranged(start, end, tries=10):
    """Fetch a byte range with a short timeout and retries (the server resets long streams)."""
    for _ in range(tries):
        try:
            r = requests.get(URL, headers={"Range": f"bytes={start}-{end}"}, timeout=(10, 30))
            if r.status_code in (200, 206) and len(r.content) == end - start + 1:
                return r.content
        except Exception:
            pass
        time.sleep(1.5)
    raise RuntimeError(f"range failed {start}-{end}")


def fetch_member(name, chunk=1 << 20):
    """Download one zip member by small ranged requests and inflate it."""
    info = next(i for i in infos if i.filename == name)
    out = RT / Path(name).name
    hdr = ranged(info.header_offset, info.header_offset + 29)
    nlen, elen = struct.unpack("<HH", hdr[26:30])
    pos = info.header_offset + 30 + nlen + elen
    last = pos + info.compress_size - 1
    comp = bytearray()
    while pos <= last:
        end = min(pos + chunk - 1, last)
        comp += ranged(pos, end)
        pos = end + 1
    raw = zlib.decompress(bytes(comp), -15) if info.compress_type == 8 else bytes(comp)
    out.write_bytes(raw)
    return out


try:
    loc_name = next(i.filename for i in infos if i.filename.endswith("Localisation/0_videos_results.json"))
    info0 = next(i for i in infos if i.filename == loc_name)
    print("compressed MB:", round(info0.compress_size / 2**20, 1), "| raw MB:", round(info0.file_size / 2**20, 1))
    t0 = time.time()
    p = fetch_member(loc_name)
    print("saved", p.name, round(p.stat().st_size / 2**20), "MB in", round(time.time() - t0), "s")
    dl = json.loads(p.read_text())
    print(type(dl).__name__, "| size:", len(dl))
    ks = list(dl.keys()) if isinstance(dl, dict) else None
    v = dl[ks[0]] if ks else dl[0]
    print("first keys:", ks[:5] if ks else None)
    print(type(v).__name__, json.dumps(v)[:1200])
except Exception:
    print(traceback.format_exc()[-1200:])


Counts the annotation categories and label fields in the RoadText localisation JSON (for example `English_Legible`) to find the legible-text boxes.

In [ ]:
from collections import Counter

cats = Counter()
label_keys = Counter()
example = {}
for fr in dl:
    for lb in fr.get("labels") or []:
        cats[lb.get("category")] += 1
        label_keys.update(lb.keys())
        example.setdefault(lb.get("category"), lb)
print("frames:", len(dl), "| videos:", len({fr["videoName"] for fr in dl}))
print("categories:", dict(cats))
print("label keys:", dict(label_keys))
for k, v in list(example.items())[:5]:
    print(k, "->", json.dumps(v)[:420])


Joins the legibility attribute with the transcription files to build a table of text tracks: video, track id, text and box size.

In [ ]:
import pandas as pd

attr = Counter()
for fr in dl:
    for lb in fr.get("labels") or []:
        if lb["category"] == "English_Legible":
            attr[tuple(lb["attributes"]["English_Legible"])] += 1
print("English_Legible attribute values:", dict(attr))

tname = next(i.filename for i in infos if i.filename.endswith("Text_0_videos_results.json.json"))
trans = json.loads(fetch_member(tname).read_text())
print("transcription videos:", len(trans), "| sample keys:", list(trans)[:6])

rows = []
for fr in dl:
    for lb in fr.get("labels") or []:
        a = lb["attributes"]["English_Legible"]
        if lb["category"] == "English_Legible" and a[1] != "English_Text":
            b = lb["box2d"]
            rows.append({"video": fr["videoName"], "frame": fr["index"], "tid": str(lb["id"]),
                         "w": b["x2"] - b["x1"], "h": b["y2"] - b["y1"],
                         "x1": b["x1"], "y1": b["y1"], "x2": b["x2"], "y2": b["y2"], "kind": a[1]})
pl = pd.DataFrame(rows)
print("plate-tagged boxes:", len(pl), "| tracks:", pl.groupby(["video", "tid"]).ngroups, "| videos:", pl.video.nunique())
print(pl.kind.value_counts().to_dict())
def text_of(r):
    return trans.get(r.video, trans.get(str(int(r.video) - 1), {})).get(r.tid)
pl["text"] = [text_of(r) for r in pl.itertuples()]
print("boxes with a transcription:", int(pl.text.notna().sum()))
trk = pl.groupby(["video", "tid"]).agg(frames=("frame", "size"), w=("w", "median"), h=("h", "median"), text=("text", "first")).reset_index()
print(trk.sort_values("w", ascending=False).head(25).round(1).to_string())


Defines `fetch_member_par()`, which downloads a whole video using 8 parallel ranged requests, and downloads the videos chosen for the validation slice (about 3 minutes).

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor


def fetch_member_par(name, chunk=2 << 20, workers=8):
    """Download one stored zip member with parallel ranged requests."""
    info = next(i for i in infos if i.filename == name)
    out = RT / Path(name).name
    if out.exists() and out.stat().st_size == info.file_size:
        return out
    hdr = ranged(info.header_offset, info.header_offset + 29)
    nlen, elen = struct.unpack("<HH", hdr[26:30])
    start = info.header_offset + 30 + nlen + elen
    last = start + info.compress_size - 1
    spans = [(s, min(s + chunk - 1, last)) for s in range(start, last + 1, chunk)]
    with ThreadPoolExecutor(workers) as ex:
        comp = b"".join(ex.map(lambda se: ranged(*se), spans))
    raw = zlib.decompress(comp, -15) if info.compress_type == 8 else comp
    out.write_bytes(raw)
    return out


def clean_len(s):
    return len(re.sub("[^a-z0-9]", "", str(s).lower()))


sel = trk[(trk.frames >= 14) & trk.text.map(clean_len).ge(6)].copy()
print("selected tracks:", len(sel), "| videos:", sorted(sel.video.unique(), key=int))
paths = {}
for v in sorted(sel.video.unique(), key=int):
    cand = [i.filename for i in infos if i.filename.endswith("/" + v + ".mp4")]
    paths[v] = cand
print({v: len(c) for v, c in paths.items()})
t0 = time.time()
for v, cand in paths.items():
    p = fetch_member_par(cand[0])
    print("video", v, round(p.stat().st_size / 2**20, 1), "MB | elapsed", round(time.time() - t0), "s", flush=True)
print("done")


Extracts crops from the downloaded videos: for each selected plate track it takes the largest boxes, keeps the sharpest frame and saves the crop. This produces the RoadText validation table `rt_val`.

In [ ]:
import cv2
import numpy as np

CROPS = RT / "crops"
CROPS.mkdir(exist_ok=True)
cand_rows = []
for v in sorted(sel.video.unique(), key=int):
    cap = cv2.VideoCapture(str(RT / (v + ".mp4")))
    print("video", v, "size", int(cap.get(3)), "x", int(cap.get(4)), "frames", int(cap.get(7)))
    want = {}
    for r in sel[sel.video == v].itertuples():
        b = pl[(pl.video == v) & (pl.tid == r.tid)].copy()
        b["area"] = b.w * b.h
        for x in b.nlargest(40, "area").itertuples():
            want.setdefault(int(x.frame), []).append((r.tid, x))
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        for tid, x in want.get(idx, []):
            pw, ph = 0.08 * (x.x2 - x.x1), 0.12 * (x.y2 - x.y1)
            x1, y1 = max(0, int(x.x1 - pw)), max(0, int(x.y1 - ph))
            x2, y2 = min(frame.shape[1], int(x.x2 + pw)), min(frame.shape[0], int(x.y2 + ph))
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            sharp = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            cand_rows.append({"video": v, "tid": tid, "frame": idx, "sharp": sharp, "w": x2 - x1, "h": y2 - y1, "img": crop})
        idx += 1
    cap.release()

cand = pd.DataFrame(cand_rows)
picked = []
for (v, tid), g in cand.groupby(["video", "tid"]):
    chosen = []
    for r in g.sort_values("sharp", ascending=False).itertuples():
        if all(abs(r.frame - c.frame) >= 15 for c in chosen):
            chosen.append(r)
        if len(chosen) == 3:
            break
    picked += chosen

out = []
for r in picked:
    path = CROPS / (r.video + "_" + r.tid + "_" + str(r.frame) + ".png")
    cv2.imwrite(str(path), r.img)
    txt = sel[(sel.video == r.video) & (sel.tid == r.tid)].text.iloc[0]
    out.append({"video": r.video, "tid": r.tid, "frame": r.frame, "w": r.w, "h": r.h, "sharp": round(r.sharp, 1),
                "text_raw": txt, "text": re.sub("[^A-Z0-9]", "", str(txt).upper()), "crop_path": str(path)})
rt_val = pd.DataFrame(out)
print("candidate crops:", len(cand), "| picked:", len(rt_val), "| tracks:", rt_val.groupby(["video", "tid"]).ngroups)
print(rt_val.groupby(["video", "tid"]).agg(text=("text_raw", "first"), n=("frame", "size"), w=("w", "median"), h=("h", "median"), sharp=("sharp", "median")).round(1).to_string())


Displays the best crop of each track with its transcription, to inspect the validation slice by eye.

In [ ]:
import matplotlib.pyplot as plt

best = rt_val.sort_values("sharp", ascending=False).groupby(["video", "tid"]).head(1).reset_index(drop=True)
fig, axes = plt.subplots(5, 3, figsize=(9, 8))
for ax in axes.ravel():
    ax.axis("off")
for ax, r in zip(axes.ravel(), best.itertuples()):
    ax.imshow(cv2.cvtColor(cv2.imread(r.crop_path), cv2.COLOR_BGR2RGB))
    ax.set_title("v" + r.video + " | " + str(r.text_raw), fontsize=9)
plt.tight_layout()
plt.show()


Saves the validation table to CSV and tries to copy it with the crops to Drive. Drive was not mounted, so it only kept a local copy; the next cell does the Drive save.

In [ ]:
import shutil

D = Path("/content/drive/MyDrive/bosch_anpr")
print("Drive mounted:", D.exists())
rt_val["source"] = "roadtext1k_us_road"
rt_val["split"] = "validation_slice"
rt_val.to_csv(RT / "roadtext_val_v1.csv", index=False)
if D.exists():
    dst = D / "data/external/roadtext_val_v1"
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RT / "roadtext_val_v1.csv", dst / "roadtext_val_v1.csv")
    shutil.make_archive(str(dst / "crops"), "zip", str(CROPS))
    print("saved to", dst, "|", sorted(p.name for p in dst.iterdir()))
else:
    print("kept locally only:", RT / "roadtext_val_v1.csv", "(mount Drive to persist)")


Mounts Drive and saves `roadtext_val_v1.csv` and a zip of the crops to `bosch_anpr/data/external/roadtext_val_v1` (38 rows).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
D = Path("/content/drive/MyDrive/bosch_anpr")
D.mkdir(parents=True, exist_ok=True)
dst = D / "data/external/roadtext_val_v1"
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(RT / "roadtext_val_v1.csv", dst / "roadtext_val_v1.csv")
shutil.make_archive(str(dst / "crops"), "zip", str(CROPS))
print("saved to", dst)
print([(p.name, round(p.stat().st_size / 1024)) for p in sorted(dst.iterdir())], "KB")
print("rows:", len(pd.read_csv(dst / "roadtext_val_v1.csv", keep_default_na=False)))


Builds the RoadText test set from the official test video blocks 700, 800 and 900: localisation and transcription tables for tracks whose text is plate-like.

In [ ]:
blocks = [700, 800, 900]          # the official RoadText-1K test videos (701-1000)
pl_rows, trans_all = [], {}
for blk in blocks:
    ln = next(i.filename for i in infos if i.filename.endswith("Localisation/" + str(blk) + "_videos_results.json"))
    tn = next(i.filename for i in infos if i.filename.endswith("Text_" + str(blk) + "_videos_results.json"))
    t0 = time.time()
    frames = json.loads(fetch_member_par(ln).read_text())
    trans_all.update(json.loads(fetch_member_par(tn).read_text()))
    for fr in frames:
        for lb in fr.get("labels") or []:
            a = lb["attributes"]["English_Legible"]
            if lb["category"] == "English_Legible" and a[1] != "English_Text":
                bx = lb["box2d"]
                pl_rows.append({"block": blk, "video": fr["videoName"], "frame": fr["index"], "tid": str(lb["id"]),
                                "x1": bx["x1"], "y1": bx["y1"], "x2": bx["x2"], "y2": bx["y2"],
                                "w": bx["x2"] - bx["x1"], "h": bx["y2"] - bx["y1"]})
    print("block", blk, "| frames", len(frames), "| elapsed", round(time.time() - t0), "s", flush=True)
    del frames

plt_ = pd.DataFrame(pl_rows)
plt_["text"] = [trans_all.get(r.video, {}).get(r.tid) for r in plt_.itertuples()]
print("plate boxes:", len(plt_), "| with direct text match:", int(plt_.text.notna().sum()), "| videos:", plt_.video.nunique())
trk_t = plt_.groupby(["block", "video", "tid"]).agg(frames=("frame", "size"), w=("w", "median"), h=("h", "median"), text=("text", "first")).reset_index()
trk_t["clean_len"] = trk_t.text.map(clean_len)
print("plate tracks:", len(trk_t), "| with text:", int(trk_t.text.notna().sum()))
cand_t = trk_t[(trk_t.frames >= 14) & trk_t.clean_len.ge(6) & trk_t.w.ge(55)].sort_values("w", ascending=False)
print("eligible tracks:", len(cand_t), "| videos:", cand_t.video.nunique())
print(cand_t.head(45).round(1).to_string())


Keeps plate-like tracks (text of at most 12 characters, one row per track) and downloads their videos (about 330 seconds).

In [ ]:
# plates only: drop long sign-like strings; keep one row per track
sel_t = cand_t[cand_t.text.map(lambda s: len(str(s)) <= 12)].copy()
print("selected tracks:", len(sel_t), "| videos:", sel_t.video.nunique(), flush=True)

t0 = time.time()
for v in sorted(sel_t.video.unique(), key=int):
    name = next(i.filename for i in infos if i.filename.endswith("/" + v + ".mp4"))
    p = fetch_member_par(name)
    print("video", v, round(p.stat().st_size / 2**20, 1), "MB | elapsed", round(time.time() - t0), "s", flush=True)
print("downloaded", flush=True)


Extracts crops for the test tracks (the 40 largest boxes per track, sharpest frame kept), producing the test table `rt_test`.

In [ ]:
CROPS_T = RT / "crops_test"
CROPS_T.mkdir(exist_ok=True)
cand_rows = []
for v in sorted(sel_t.video.unique(), key=int):
    cap = cv2.VideoCapture(str(RT / (v + ".mp4")))
    want = {}
    for r in sel_t[sel_t.video == v].itertuples():
        b = plt_[(plt_.video == v) & (plt_.tid == r.tid)].copy()
        b["area"] = b.w * b.h
        for x in b.nlargest(40, "area").itertuples():
            want.setdefault(int(x.frame), []).append((r.tid, x))
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        for tid, x in want.get(idx, []):
            pw, ph = 0.08 * (x.x2 - x.x1), 0.12 * (x.y2 - x.y1)
            x1, y1 = max(0, int(x.x1 - pw)), max(0, int(x.y1 - ph))
            x2, y2 = min(frame.shape[1], int(x.x2 + pw)), min(frame.shape[0], int(x.y2 + ph))
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            sharp = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            cand_rows.append({"video": v, "tid": tid, "frame": idx, "sharp": sharp, "w": x2 - x1, "h": y2 - y1, "img": crop})
        idx += 1
    cap.release()

cand_t2 = pd.DataFrame(cand_rows)
picked = []
for (v, tid), g in cand_t2.groupby(["video", "tid"]):
    chosen = []
    for r in g.sort_values("sharp", ascending=False).itertuples():
        if all(abs(r.frame - c.frame) >= 15 for c in chosen):
            chosen.append(r)
        if len(chosen) == 3:
            break
    picked += chosen

out = []
for r in picked:
    path = CROPS_T / (r.video + "_" + r.tid + "_" + str(r.frame) + ".png")
    cv2.imwrite(str(path), r.img)
    row = sel_t[(sel_t.video == r.video) & (sel_t.tid == r.tid)].iloc[0]
    out.append({"block": int(row.block), "video": r.video, "tid": r.tid, "frame": r.frame, "w": r.w, "h": r.h,
                "sharp": round(r.sharp, 1), "text_raw": row.text,
                "text": re.sub("[^A-Z0-9]", "", str(row.text).upper()), "crop_path": str(path),
                "source": "roadtext1k_official_test", "split": "validation_slice"})
rt_test = pd.DataFrame(out)
rt_test.to_csv(RT / "roadtext_test_v1.csv", index=False)
print("candidate crops:", len(cand_t2), "| picked:", len(rt_test), "| tracks:", rt_test.groupby(["video", "tid"]).ngroups)

D = Path("/content/drive/MyDrive/bosch_anpr")
if D.exists():
    dst = D / "data/external/roadtext_test_v1"
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RT / "roadtext_test_v1.csv", dst / "roadtext_test_v1.csv")
    shutil.make_archive(str(dst / "crops"), "zip", str(CROPS_T))
    print("saved to Drive:", [(p.name, round(p.stat().st_size / 1024)) for p in sorted(dst.iterdir())], "KB")
else:
    print("Drive not mounted; kept locally")
print(rt_test.groupby(["block", "video", "tid"]).agg(text=("text_raw", "first"), n=("frame", "size"), w=("w", "median"), sharp=("sharp", "median")).round(0).to_string())


Shows crops with unusual transcriptions (non-alphanumeric characters) next to normal ones, to review label quality in the test set.

In [ ]:
best_t = rt_test.sort_values("sharp", ascending=False).groupby(["video", "tid"]).head(1)
odd = best_t[best_t.text_raw.str.contains("[^A-Za-z0-9 -]|^[0-9 .-]+$", regex=True)]
show = pd.concat([odd.head(9), best_t[~best_t.index.isin(odd.index)].head(9)])
fig, axes = plt.subplots(6, 3, figsize=(9, 9))
for ax in axes.ravel():
    ax.axis("off")
for ax, r in zip(axes.ravel(), show.itertuples()):
    ax.imshow(cv2.cvtColor(cv2.imread(r.crop_path), cv2.COLOR_BGR2RGB))
    ax.set_title("v" + r.video + " | " + str(r.text_raw), fontsize=9)
plt.tight_layout()
plt.show()


**Status check after the runtime was recycled.** Drive, the local crops and the code are gone, there is no GPU, and `rt_test` is undefined. The next cell restores the state.

In [ ]:
import os
from pathlib import Path

print("drive:", Path("/content/drive/MyDrive/bosch_anpr").exists(),
      "| local crops_test:", Path("/content/ext/roadtext/crops_test").exists(),
      "| ANPR code:", Path("/content/ANPR/src/anpr/ocr").exists())
import subprocess
print(subprocess.run("nvidia-smi --query-gpu=name,memory.used --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip() or "no GPU")
try:
    print("rt_test rows:", len(rt_test), "| rt_val rows:", len(rt_val))
except NameError as e:
    print("variables gone:", e)


Restores the state: mounts Drive, copies the code from `code_backup`, reloads the saved RoadText validation table (143 crops from 56 tracks).

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
D = Path("/content/drive/MyDrive/bosch_anpr")
REPO = Path("/content/ANPR")
if not (REPO / "src/anpr/ocr").exists():
    REPO.mkdir(exist_ok=True)
    shutil.copytree(D / "code_backup/src", REPO / "src", dirs_exist_ok=True)
sys.path.insert(0, str(REPO / "src"))

RT = Path("/content/ext/roadtext")
RT.mkdir(parents=True, exist_ok=True)
frames = []
for tag, folder in [("roadtext_val_v1", "crops"), ("roadtext_test_v1", "crops_test")]:
    src = D / "data/external" / tag
    with zipfile.ZipFile(src / "crops.zip") as z:
        z.extractall(RT / folder)
    df = pd.read_csv(src / (tag + ".csv"), keep_default_na=False, dtype={"video": str, "tid": str})
    df["set"] = tag
    frames.append(df)
rt = pd.concat(frames, ignore_index=True)
rt["exists"] = rt.crop_path.map(lambda p: Path(p).exists())
print("rows:", len(rt), "| crops found:", int(rt.exists.sum()), "| tracks:", rt.groupby(["video", "tid"]).ngroups)


## Part 5: OCR baseline, pretrained SVTRv2 vs a vision-language teacher

Runs the pretrained SVTRv2 recogniser (OpenOCR, server mode) on the 143 RoadText crops as the baseline reader. Takes 6 seconds.

In [ ]:
!pip install -q openocr-python 2>&1 | tail -2
import time

from openocr import OpenOCR

from anpr.ocr.consensus import normalize_plate

reader = OpenOCR(task="rec", mode="server", backend="torch")
t0 = time.time()
preds = []
for p in rt.crop_path:
    r = reader(image_path=p)
    preds.append({"s_text": normalize_plate(r[0]["text"]), "s_score": float(r[0]["score"])})
rt = rt.drop(columns=["s_text", "s_score"], errors="ignore").join(pd.DataFrame(preds))
rt.to_csv("/content/rt_preds_svtr.csv", index=False)
print("SVTRv2 done:", len(preds), "crops in", round(time.time() - t0), "s")


Loads the Qwen3-VL teacher model in 4-bit (`anpr.ocr.teacher`) with a generic licence-plate prompt, and transcribes the same 143 crops. Total about 6 minutes, of which 297 seconds is loading the model.

In [ ]:
!pip install -q -U bitsandbytes accelerate
import time

import torch

import anpr.ocr.teacher as T

assert torch.cuda.is_available(), "no GPU: switch the runtime type to T4"
# these crops are not Indian plates, so use a generic prompt
T.PROMPT = T.PROMPT.replace("an Indian vehicle number plate", "a vehicle license plate")
t0 = time.time()
model, processor = T.load_teacher()
print("loaded in", round(time.time() - t0), "s")
res = T.transcribe(model, processor, rt.crop_path.tolist(), batch_size=8)
tp = pd.DataFrame(res).add_prefix("t_")
rt = rt.drop(columns=[c for c in rt.columns if c.startswith("t_")], errors="ignore").join(tp)
rt.to_csv(D / "data/external/roadtext_preds_v1.csv", index=False)
print("TEACHER_DONE", len(res), "crops | total", round(time.time() - t0), "s")


**Failed attempt.** Scores both readers against the ground truth (exact match and character error rate per crop and per track, using voting). It stops with `AttributeError`: `rt.gt` refers to pandas' built-in `gt` method instead of the new column named `gt`, so the column must be read as `rt['gt']`.

In [ ]:
import traceback
try:
    import numpy as np

    from anpr.ocr.consensus import cer, vote

    rt["gt"] = rt.text
    rt["digits_only"] = rt.gt.str.isdigit()
    readers = {"SVTRv2": "s_text", "Teacher": "t_text"}


    def crop_scores(df, col):
        ex = (df[col] == df.gt).mean()
        c = np.mean([cer(p, g) for p, g in zip(df[col], df.gt)])
        return round(100 * ex, 1), round(float(c), 3)


    def per_track(df, col):
        rows = []
        for (v, tid), g in df.groupby(["video", "tid"]):
            pred, _ = vote({str(i): p for i, p in enumerate(g[col])})
            rows.append({"video": v, "tid": tid, "gt": g.gt.iloc[0], "pred": pred, "digits_only": g.digits_only.iloc[0]})
        return pd.DataFrame(rows)


    print("crops:", len(rt), "| tracks:", rt.groupby(["video", "tid"]).ngroups, "| unique plate texts:", rt.gt.nunique())
    print("empty reads:", {n: int(rt[c].eq("").sum()) for n, c in readers.items()})
    print()
    print("CROP level  (exact %, mean CER):")
    for n, c in readers.items():
        print("  ", n, crop_scores(rt, c), "| digits-only plates:", crop_scores(rt[rt.digits_only], c), "| letter+digit plates:", crop_scores(rt[~rt.digits_only], c))
    print("TRACK level (majority vote over the track's crops):")
    tracks = {}
    for n, c in readers.items():
        tr = per_track(rt, c)
        tracks[n] = tr
        uniq = tr.drop_duplicates("gt")
        print("  ", n, "tracks exact %:", round(100 * (tr.pred == tr.gt).mean(), 1), "| unique plates exact %:", round(100 * (uniq.pred == uniq.gt).mean(), 1), "of", len(uniq))
    print()
    both = tracks["SVTRv2"].merge(tracks["Teacher"], on=["video", "tid", "gt"], suffixes=("_s", "_t"))
    sus = both[(both.pred_s == both.pred_t) & both.pred_s.ne("") & both.pred_s.ne(both.gt)]
    print("tracks where both readers agree with each other but not with the annotation (suspect labels):", len(sus))
    print(sus[["video", "tid", "gt", "pred_s"]].to_string())
    hard = both[(both.pred_s != both.gt) & (both.pred_t != both.gt) & ~both.index.isin(sus.index)]
    print("both wrong and disagree with each other:", len(hard))
    print(hard[["video", "tid", "gt", "pred_s", "pred_t"]].head(14).to_string())

except Exception:
    print(traceback.format_exc()[-1500:])


## Part 6: Detector tooling and data

New session: checks the GPU (Tesla T4), mounts Drive and confirms the project folder exists.

In [ ]:
import subprocess

from pathlib import Path

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "no GPU")
from google.colab import drive

drive.mount("/content/drive")
print("drive ok:", Path("/content/drive/MyDrive/bosch_anpr").exists())


Checks the saved OCR models on Drive (`ocr_repsvtr_ft_v1` and `ocr_repsvtr_ft_v2`), restores the code and shows the Kaggle YOLO-format data config (`/content/data/yolo`, one class: `license_plate`).

In [ ]:
import collections
import shutil
import sys
import tarfile
import traceback

D = Path("/content/drive/MyDrive/bosch_anpr")
REPO = Path("/content/ANPR")
try:
    for m in ["ocr_repsvtr_ft_v1", "ocr_repsvtr_ft_v2"]:
        p = D / "models" / m
        print(m, sorted(f.name for f in p.glob("*")) if p.exists() else "MISSING", [round(f.stat().st_size / 1e6, 1) for f in p.glob("*.pt")], "MB")
    if not (REPO / "src/anpr").exists():
        REPO.mkdir(exist_ok=True)
        shutil.copytree(D / "code_backup/src", REPO / "src", dirs_exist_ok=True)
        shutil.copytree(D / "code_backup/tests", REPO / "tests", dirs_exist_ok=True)
    sys.path.insert(0, str(REPO / "src"))
    print("code modules:", sorted(p.name for p in (REPO / "src/anpr").iterdir() if p.is_dir()), "| ocr files:", sorted(p.name for p in (REPO / "src/anpr/ocr").glob("*.py")))
    LOCAL = Path("/content/data")
    LOCAL.mkdir(exist_ok=True)
    shutil.copy2(D / "data/processed/anpr_yolo_v1.tar", "/content/yolo.tar")
    with tarfile.open("/content/yolo.tar") as t:
        t.extractall(LOCAL, filter="data")
    shutil.copy2(D / "data/raw/kaggle_indian_plates_raw.tar", "/content/raw.tar")
    with tarfile.open("/content/raw.tar") as t:
        t.extractall(LOCAL, filter="data")
    print("top:", sorted(p.name for p in LOCAL.iterdir()))
    for sub in sorted(LOCAL.iterdir()):
        if sub.is_dir():
            files = [f for f in sub.rglob("*") if f.is_file()]
            print(sub.name, len(files), collections.Counter(f.suffix.lower() for f in files).most_common(4), "| subdirs:", sorted({str(f.parent.relative_to(sub)) for f in files})[:8])
    for y in LOCAL.rglob("*.yaml"):
        print("yaml:", y, "->", y.read_text()[:400].replace(chr(10), " | "))
    print((D / "data/processed/manifest_v1.csv").exists(), (D / "data/processed/DATA_VERSION.json").exists())
except Exception:
    print(traceback.format_exc()[-1500:])


Defines the `sh()` shell helper, prints library versions and installs `ultralytics` and `rfdetr[train]` for the detector experiments; lists the project's modules.

In [ ]:
import subprocess

def sh(cmd, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout + r.stderr

print(sh("python -c 'import torch; print(torch.__version__)'; pip list 2>/dev/null | grep -i -E '^(torch|torchvision|numpy|ultralytics|rfdetr|supervision|pycocotools) '"))
print(sh("pip install -q ultralytics 2>&1 | tail -3"))
print(sh("pip install -q 'rfdetr[train]' 2>&1 | tail -5"))
print(sh("python -c 'import torch, torchvision, ultralytics, rfdetr; print(torch.__version__, torchvision.__version__, ultralytics.__version__, getattr(rfdetr, \"__version__\", \"?\"))' 2>&1 | tail -4"))
print(sh("ls src/anpr/eval src/anpr/models src/anpr/deploy src/anpr/data", "/content/ANPR"))


Writes `anpr/eval/detection.py`: single-class detection metrics (AP averaged over IoU thresholds 0.5 to 0.95, and recall by object-size bin) plus tests. Its 4 tests pass; the lint step failed only because ruff was not installed yet.

In [ ]:
(REPO / "src/anpr/eval/detection.py").write_text('''"""Single-class detection metrics: AP over IoU thresholds and recall by object size."""
from __future__ import annotations

import numpy as np

IOU_THRESHOLDS = np.round(np.arange(0.5, 0.96, 0.05), 2)
SIZE_BINS = ((0.0, 0.005, "<0.5%"), (0.005, 0.02, "0.5-2%"), (0.02, 0.10, "2-10%"), (0.10, 1.01, ">=10%"))


def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between boxes a (N, 4) and b (M, 4) in xyxy format."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(axis=2)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)


def _match(preds, gts, iou_thr, score_thr=0.0):
    """Greedy matching per image; returns scores, tp flags and per-GT matched flags."""
    scores, tps, gt_hit = [], [], []
    for (pb, ps), gb in zip(preds, gts):
        keep = np.asarray(ps) >= score_thr
        pb, ps = np.asarray(pb).reshape(-1, 4)[keep], np.asarray(ps)[keep]
        order = np.argsort(-ps)
        pb, ps = pb[order], ps[order]
        ious = iou_matrix(pb, gb)
        taken = np.zeros(len(gb), dtype=bool)
        for i in range(len(pb)):
            tp = False
            if len(gb):
                cand = np.where(~taken, ious[i], -1.0)
                j = int(cand.argmax())
                if cand[j] >= iou_thr:
                    taken[j] = True
                    tp = True
            scores.append(ps[i])
            tps.append(tp)
        gt_hit.append(taken)
    return np.array(scores), np.array(tps, dtype=bool), gt_hit


def average_precision(preds, gts, iou_thr: float = 0.5) -> float:
    """COCO-style 101-point AP for one class."""
    n_gt = sum(len(g) for g in gts)
    scores, tps, _ = _match(preds, gts, iou_thr)
    if n_gt == 0 or len(scores) == 0:
        return 0.0
    order = np.argsort(-scores, kind="stable")
    tp = np.cumsum(tps[order])
    fp = np.cumsum(~tps[order])
    recall = tp / n_gt
    precision = tp / np.maximum(tp + fp, 1e-9)
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    grid = np.linspace(0, 1, 101)
    idx = np.searchsorted(recall, grid, side="left")
    vals = np.where(idx < len(precision), precision[np.minimum(idx, len(precision) - 1)], 0.0)
    return float(vals.mean())


def evaluate(preds, gts, image_areas, score_thr: float = 0.25) -> dict:
    """AP50, AP75, AP50-95, precision/recall at a score threshold and recall by size bin.

    Args:
        preds: per image, a tuple (boxes (N, 4) xyxy in pixels, scores (N,)).
        gts: per image, ground-truth boxes (M, 4) xyxy in pixels.
        image_areas: per image, width * height in pixels (for size bins).
        score_thr: confidence threshold for precision, recall and size-bin recall.
    """
    gts = [np.asarray(g, dtype=np.float64).reshape(-1, 4) for g in gts]
    aps = [average_precision(preds, gts, t) for t in IOU_THRESHOLDS]
    scores, tps, gt_hit = _match(preds, gts, 0.5, score_thr)
    n_gt = sum(len(g) for g in gts)
    n_hit = int(sum(h.sum() for h in gt_hit))
    out = {
        "n_images": len(gts),
        "n_gt": n_gt,
        "AP50": round(aps[0], 4),
        "AP75": round(aps[5], 4),
        "AP50-95": round(float(np.mean(aps)), 4),
        "precision": round(float(tps.sum()) / max(len(tps), 1), 4),
        "recall": round(n_hit / max(n_gt, 1), 4),
        "score_thr": score_thr,
    }
    max_rec = sum(int(h.sum()) for h in _match(preds, gts, 0.5, 0.0)[2]) / max(n_gt, 1)
    out["recall_any_conf"] = round(max_rec, 4)
    frac = np.concatenate([(g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1]) / a for g, a in zip(gts, image_areas)]) if n_gt else np.zeros(0)
    hit = np.concatenate([h for h in gt_hit]) if n_gt else np.zeros(0, dtype=bool)
    for lo, hi, name in SIZE_BINS:
        m = (frac >= lo) & (frac < hi)
        out["recall " + name] = (round(float(hit[m].mean()), 3) if m.any() else None, int(m.sum()))
    return out
''')

(REPO / "tests/test_detection_metrics.py").write_text('''import numpy as np

from anpr.eval.detection import average_precision, evaluate, iou_matrix


def _box(x1, y1, x2, y2):
    return np.array([[x1, y1, x2, y2]], dtype=float)


def test_iou_identical_and_disjoint():
    assert np.isclose(iou_matrix(_box(0, 0, 10, 10), _box(0, 0, 10, 10))[0, 0], 1.0)
    assert iou_matrix(_box(0, 0, 10, 10), _box(20, 20, 30, 30))[0, 0] == 0.0


def test_perfect_prediction_gives_ap_one():
    gts = [_box(10, 10, 50, 30)]
    preds = [(_box(10, 10, 50, 30), np.array([0.9]))]
    assert average_precision(preds, gts, 0.5) == 1.0


def test_no_prediction_gives_zero_and_false_positive_lowers_ap():
    gts = [_box(10, 10, 50, 30)]
    assert average_precision([(np.zeros((0, 4)), np.zeros(0))], gts, 0.5) == 0.0
    boxes = np.vstack([_box(200, 200, 240, 220), _box(10, 10, 50, 30)])
    preds = [(boxes, np.array([0.95, 0.9]))]
    assert 0.0 < average_precision(preds, gts, 0.5) < 1.0


def test_evaluate_reports_size_bins():
    gts = [_box(0, 0, 10, 10), _box(0, 0, 500, 500)]
    preds = [(_box(0, 0, 10, 10), np.array([0.8])), (np.zeros((0, 4)), np.zeros(0))]
    res = evaluate(preds, gts, [10000.0, 1000000.0])
    assert res["recall 0.5-2%"][0] == 1.0 and res["recall >=10%"][0] == 0.0 or res["recall <0.5%"] is not None
    assert res["n_gt"] == 2
''')

print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -8; PYTHONPATH=src python -m pytest -q tests/test_detection_metrics.py 2>&1 | tail -8"))


Installs ruff and pytest, formats and lints the code and runs all tests: lint is clean and 23 tests pass.

In [ ]:
import re

print(sh("pip install -q ruff pytest 2>&1 | tail -1"))
pp = REPO / "pyproject.toml"
if not pp.exists():
    pp.write_text("[tool.ruff]\nline-length = 88\nsrc = ['src']\n[tool.ruff.lint]\nselect = ['E', 'F', 'W', 'D', 'I']\nignore = ['D203', 'D213', 'E501']\n[tool.ruff.lint.pydocstyle]\nconvention = 'google'\n[tool.ruff.lint.per-file-ignores]\n'tests/*' = ['D']\n")
tp = REPO / "tests/test_detection_metrics.py"
s = tp.read_text()
s = re.sub(r'    assert res\["recall 0.5-2%"\]\[0\] == 1.0 and res\["recall >=10%"\]\[0\] == 0.0 or res\["recall <0.5%"\] is not None', '    assert res["recall 0.5-2%"][0] == 1.0 and res["recall >=10%"][0] == 0.0', s)
tp.write_text(s)
print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -12; PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -5"))


Builds the Saisirishan evaluation set (boxes, splits, near-duplicate removal with perceptual hashes) and reports box statistics: the median plate covers about 81% of the image on Saisirishan and 77% on the Kaggle test set, so most plates are close-ups.

In [ ]:
import json
import os
import traceback
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd

print(sh("pip install -q imagehash kaggle huggingface_hub 2>&1 | tail -1"))
import imagehash
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
from PIL import Image

disable_progress_bars()
try:
    from google.colab import userdata

    SS = Path("/content/ext/saisirishan")
    SS.mkdir(parents=True, exist_ok=True)
    if not any(SS.rglob("*.xml")):
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        subprocess.run(["kaggle", "datasets", "download", "-d", "saisirishan/indian-vehicle-dataset", "-p", str(SS), "--unzip"], capture_output=True)
    DC = Path("/content/ext/datacluster")
    snapshot_download("Dataclusterlabspvtltd/indian-number-plates-dataset", repo_type="dataset", local_dir=str(DC))

    def find_img(xml_path, root):
        d = xml_path.parent
        names = [root.findtext("filename") or "", xml_path.name[:-4], xml_path.name[:-8]]
        for n in names:
            if n and (d / n).is_file():
                return d / n
        for n in names:
            stem = Path(n).stem if n else ""
            for e in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
                if stem and (d / (stem + e)).is_file():
                    return d / (stem + e)
        return None

    def parse(root_dir, name, need_digit):
        rows = []
        for p in sorted(root_dir.rglob("*.xml")):
            root = ET.parse(p).getroot()
            img = find_img(p, root)
            if img is None:
                img = next((q for q in [p.with_suffix(".jpg"), p.with_suffix(".jpeg"), p.with_suffix(".png"), p.parent.parent / "images" / (p.stem + ".jpg")] if q.is_file()), None)
            if img is None:
                continue
            boxes = []
            for obj in root.iter("object"):
                bb = obj.find("bndbox")
                if bb is None:
                    continue
                txt = obj.findtext("name") or ""
                if need_digit and not any(ch.isdigit() for ch in txt):
                    continue
                boxes.append([float(bb.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")])
            if not boxes:
                continue
            im = cv2.imread(str(img))
            if im is None:
                continue
            h, w = im.shape[:2]
            rows.append({"set": name, "folder": p.relative_to(root_dir).parts[0], "image": str(img), "w": w, "h": h, "boxes": boxes})
        return pd.DataFrame(rows)

    sai = parse(SS, "saisirishan", True)
    dcd = parse(DC, "datacluster", False)
    print("saisirishan images:", len(sai), "boxes:", int(sai.boxes.map(len).sum()), "| datacluster images:", len(dcd), "boxes:", int(dcd.boxes.map(len).sum()), flush=True)

    det = pd.read_csv(D / "data/processed/manifest_v1.csv", keep_default_na=False)
    def ph(path):
        try:
            return np.array(imagehash.phash(Image.open(path)).hash.flatten())
        except Exception:
            return None

    kh = np.array([h for h in (ph(p) for p in det.drop_duplicates("group").image) if h is not None])
    for df in (sai, dcd):
        df["overlap"] = [bool((lambda h: h is not None and ((kh != h).sum(axis=1) <= 6).any())(ph(p))) for p in df.image]
    print("overlap with Kaggle images -> saisirishan:", int(sai.overlap.sum()), "| datacluster:", int(dcd.overlap.sum()))
    sai, dcd = sai[~sai.overlap].reset_index(drop=True), dcd[~dcd.overlap].reset_index(drop=True)

    def stats(df, name):
        fr = np.concatenate([[(b[2] - b[0]) * (b[3] - b[1]) / (r.w * r.h) for b in r.boxes] for r in df.itertuples()])
        print(name, "| images", len(df), "| boxes", len(fr), "| median plate area %.2f%% of image" % (100 * np.median(fr)), "| share <2%%: %.0f%%" % (100 * (fr < 0.02).mean()),
              "| multi-plate images:", int((df.boxes.map(len) > 1).sum()), "| median size", int(df.w.median()), "x", int(df.h.median()))
    stats(sai, "saisirishan")
    print(sai.groupby("folder").size().to_dict())
    stats(dcd, "datacluster")
    OUT = D / "data/external/det_eval_v1"
    OUT.mkdir(parents=True, exist_ok=True)
    for nm, df in (("saisirishan", sai), ("datacluster", dcd)):
        d2 = df.copy()
        d2["boxes"] = d2.boxes.map(json.dumps)
        d2["image"] = d2.image.map(lambda s: Path(s).name)
        d2.to_csv(OUT / (nm + "_boxes.csv"), index=False)
    print("saved box tables (no images) to", OUT)
    # Kaggle YOLO splits: boxes per split and plate size
    for sp in ["train", "val", "test"]:
        labs = list((Path("/content/data/yolo") / sp / "labels").glob("*.txt"))
        wh = [list(map(float, ln.split()[3:5])) for f in labs for ln in f.read_text().splitlines() if ln.strip()]
        areas = np.array([a * b for a, b in wh])
        print("kaggle", sp, "| images", len(labs), "| boxes", len(areas), "| median area %.1f%%" % (100 * np.median(areas)), "| share <2%%: %.0f%%" % (100 * (areas < 0.02).mean()))
except Exception:
    print(traceback.format_exc()[-2000:])


Downloads COCO val2017 images and annotations (5,000 images). They are used as real vehicle backgrounds for the composite training data.

In [ ]:
import time
import zipfile

CO = Path("/content/coco")
CO.mkdir(exist_ok=True)
t0 = time.time()
print(sh("wget -q -O /content/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip && wget -q -O /content/coco/ann.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip; ls -la /content/coco"))
with zipfile.ZipFile(CO / "val2017.zip") as z:
    z.extractall(CO)
with zipfile.ZipFile(CO / "ann.zip") as z:
    z.extract("annotations/instances_val2017.json", CO)
print("COCO ready in", round(time.time() - t0), "s |", len(list((CO / "val2017").glob("*.jpg"))), "images")


Writes `anpr/data/composite.py`, which pastes plates onto real COCO vehicle photos at realistic scale to create training images with small plates, plus its tests (2 pass).

In [ ]:
(REPO / "src/anpr/data/composite.py").write_text('''"""Paste plates onto real vehicle photos to create small-plate detection training data."""
from __future__ import annotations

import json
import random
from pathlib import Path

import cv2
import numpy as np

from anpr.ocr.synth import _occlude, degrade, random_plate_text, render_plate

VEHICLE_CATS = {3: "car", 4: "motorcycle", 6: "bus", 8: "truck"}


def load_vehicles(ann_path: Path, min_width: float = 120.0) -> dict[str, list[tuple[int, list[float]]]]:
    """Read COCO annotations and return large vehicle boxes (xyxy) per file name."""
    data = json.loads(Path(ann_path).read_text())
    names = {im["id"]: im["file_name"] for im in data["images"]}
    out: dict[str, list[tuple[int, list[float]]]] = {n: [] for n in names.values()}
    for a in data["annotations"]:
        if a["category_id"] in VEHICLE_CATS and not a.get("iscrowd", 0) and a["bbox"][2] >= min_width:
            x, y, w, h = a["bbox"]
            out[names[a["image_id"]]].append((a["category_id"], [x, y, x + w, y + h]))
    return out


class PlatePool:
    """Source of plate images: real Kaggle crops mixed with synthetic plates."""

    def __init__(self, real: list[np.ndarray], synth_frac: float = 0.6):
        """Store real crops (BGR arrays) and the share of synthetic plates."""
        self.real = real
        self.synth_frac = synth_frac

    def sample(self, rng: random.Random, two_line: bool | None = None) -> np.ndarray:
        """Return one plate image, optionally preferring a two-line layout."""
        if self.real and rng.random() > self.synth_frac:
            return rng.choice(self.real)
        for _ in range(10):
            img = render_plate(random_plate_text(rng), rng)
            is_two = img.shape[1] / img.shape[0] < 2.2
            if two_line is None or is_two == two_line:
                break
        return degrade(img, rng) if rng.random() < 0.7 else img


def paste_plate(img: np.ndarray, vbox: list[float], cat: int, plate: np.ndarray, rng: random.Random):
    """Paste a plate on a vehicle with perspective, blur and feathering; return its xyxy box or None."""
    x1, y1, x2, y2 = vbox
    vw, vh = x2 - x1, y2 - y1
    moto = cat == 4
    pw = vw * (rng.uniform(0.12, 0.28) if moto else rng.uniform(0.14, 0.32))
    cx = x1 + vw * rng.uniform(0.35, 0.65)
    cy = y1 + vh * (rng.uniform(0.55, 0.85) if moto else rng.uniform(0.68, 0.90))
    ph = pw * plate.shape[0] / plate.shape[1]
    tw, th = max(int(pw), 8), max(int(ph), 5)
    patch = cv2.resize(plate, (tw, th), interpolation=cv2.INTER_AREA)
    if rng.random() < 0.1:
        patch = _occlude(patch, rng)
    alpha = np.full((th, tw), 255, np.uint8)
    j = rng.uniform(0.0, 0.08)
    src = np.float32([[0, 0], [tw, 0], [tw, th], [0, th]])
    dst = src + np.float32([[rng.uniform(-j, j) * tw, rng.uniform(-j, j) * th] for _ in range(4)])
    pad = 4
    dst += pad
    size = (tw + 2 * pad, th + 2 * pad)
    mat = cv2.getPerspectiveTransform(src, dst)
    patch = cv2.warpPerspective(patch, mat, size, borderMode=cv2.BORDER_REPLICATE)
    alpha = cv2.warpPerspective(alpha, mat, size)
    if rng.random() < 0.7:
        patch = cv2.GaussianBlur(patch, (0, 0), rng.uniform(0.3, 1.3))
    alpha = cv2.GaussianBlur(alpha, (0, 0), 0.8).astype(np.float32)[..., None] / 255.0
    x0, y0 = int(cx - size[0] / 2), int(cy - size[1] / 2)
    H, W = img.shape[:2]
    if x0 < 0 or y0 < 0 or x0 + size[0] > W or y0 + size[1] > H:
        return None
    region = img[y0 : y0 + size[1], x0 : x0 + size[0]].astype(np.float32)
    local = region.mean()
    patch = patch.astype(np.float32)
    patch *= float(np.clip(local / max(patch.mean(), 1.0), 0.6, 1.5)) ** 0.5
    img[y0 : y0 + size[1], x0 : x0 + size[0]] = np.clip(alpha * patch + (1 - alpha) * region, 0, 255).astype(np.uint8)
    ys, xs = np.where(alpha[..., 0] > 0.5)
    return [float(x0 + xs.min()), float(y0 + ys.min()), float(x0 + xs.max() + 1), float(y0 + ys.max() + 1)]


def make_composite(img: np.ndarray, vehicles: list[tuple[int, list[float]]], pool: PlatePool, rng: random.Random, p_negative: float = 0.12):
    """Return a copy of img with plates pasted on up to three vehicles, and their boxes."""
    out = img.copy()
    boxes: list[list[float]] = []
    if not vehicles or rng.random() < p_negative:
        return out, boxes
    chosen = rng.sample(vehicles, k=min(len(vehicles), rng.choice([1, 1, 2, 3])))
    for cat, vb in chosen:
        plate = pool.sample(rng, two_line=True if cat == 4 and rng.random() < 0.5 else None)
        box = paste_plate(out, vb, cat, plate, rng)
        if box is not None:
            boxes.append(box)
    if boxes and rng.random() < 0.5:
        ok, enc = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, rng.randint(40, 92)])
        out = cv2.imdecode(enc, cv2.IMREAD_COLOR) if ok else out
    return out, boxes


def build_split(coco_dir: Path, names: list[str], vehicles: dict, pool: PlatePool, out_dir: Path, n: int, seed: int = 0) -> int:
    """Write n composite images with YOLO labels under out_dir/images and out_dir/labels."""
    rng = random.Random(seed)
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    with_veh = [m for m in names if vehicles.get(m)]
    written = 0
    for i in range(n):
        name = rng.choice(with_veh) if rng.random() < 0.95 else rng.choice(names)
        img = cv2.imread(str(Path(coco_dir) / name))
        if img is None:
            continue
        comp, boxes = make_composite(img, vehicles.get(name, []), pool, rng)
        h, w = comp.shape[:2]
        stem = "c" + str(i).zfill(5)
        cv2.imwrite(str(out_dir / "images" / (stem + ".jpg")), comp)
        lines = [
            "0 %.6f %.6f %.6f %.6f" % ((b[0] + b[2]) / 2 / w, (b[1] + b[3]) / 2 / h, (b[2] - b[0]) / w, (b[3] - b[1]) / h)
            for b in boxes
        ]
        (out_dir / "labels" / (stem + ".txt")).write_text(chr(10).join(lines) + (chr(10) if lines else ""))
        written += 1
    return written
''')

(REPO / "tests/test_composite.py").write_text('''import random

import numpy as np

from anpr.data.composite import PlatePool, make_composite


def test_composite_pastes_plate_inside_image():
    rng = random.Random(0)
    img = np.full((480, 640, 3), 120, np.uint8)
    vehicles = [(3, [100.0, 100.0, 500.0, 420.0])]
    pool = PlatePool([], synth_frac=1.0)
    found = 0
    for _ in range(20):
        out, boxes = make_composite(img, vehicles, pool, rng, p_negative=0.0)
        assert out.shape == img.shape
        for x1, y1, x2, y2 in boxes:
            assert 0 <= x1 < x2 <= 640 and 0 <= y1 < y2 <= 480
            found += 1
    assert found > 0


def test_no_vehicles_means_no_plates():
    rng = random.Random(1)
    out, boxes = make_composite(np.zeros((100, 100, 3), np.uint8), [], PlatePool([]), rng)
    assert boxes == []
''')

print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -12; PYTHONPATH=src python -m pytest -q tests/test_composite.py 2>&1 | tail -8"))


Generates the first composite dataset (3,000 training and 400 validation images) from COCO vehicles and a pool of real and synthetic plates. Takes 50 seconds.

In [ ]:
import importlib

import anpr.data.composite as _c
import anpr.ocr.synth as _s

importlib.reload(_s)
importlib.reload(_c)
import random
import time
import traceback

import cv2
import numpy as np

from anpr.data.composite import PlatePool, build_split, load_vehicles

try:
    t0 = time.time()
    Y = Path("/content/data/yolo")
    real = []
    for lab_file in sorted((Y / "train/labels").glob("*.txt")):
        stem = lab_file.stem
        imgp = next((p for p in [Y / "train/images" / (stem + e) for e in (".png", ".jpg")] if p.exists()), None)
        if imgp is None:
            continue
        im = cv2.imread(str(imgp))
        if im is None:
            continue
        H, W = im.shape[:2]
        for ln in lab_file.read_text().splitlines():
            if not ln.strip():
                continue
            _, cx, cy, bw, bh = map(float, ln.split()[:5])
            x1, y1 = max(0, int((cx - bw / 2 - 0.03 * bw) * W)), max(0, int((cy - bh / 2 - 0.05 * bh) * H))
            x2, y2 = min(W, int((cx + bw / 2 + 0.03 * bw) * W)), min(H, int((cy + bh / 2 + 0.05 * bh) * H))
            if x2 - x1 < 20 or y2 - y1 < 8:
                continue
            crop = im[y1:y2, x1:x2]
            s = min(1.0, 300 / crop.shape[1])
            real.append(cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else crop.copy())
    print("real plate crops in pool:", len(real), "| median aspect %.2f" % np.median([c.shape[1] / c.shape[0] for c in real]), flush=True)
    CO = Path("/content/coco")
    veh = load_vehicles(CO / "annotations/instances_val2017.json")
    names = sorted(veh)
    cut = int(0.8 * len(names))
    tr_names, va_names = names[:cut], names[cut:]
    print("coco images:", len(names), "| with vehicles train/val bg:", sum(bool(veh[n]) for n in tr_names), sum(bool(veh[n]) for n in va_names), flush=True)
    pool = PlatePool(real, synth_frac=0.6)
    COMP = Path("/content/composite")
    n_tr = build_split(CO / "val2017", tr_names, veh, pool, COMP / "train", 3000, seed=1)
    n_va = build_split(CO / "val2017", va_names, veh, pool, COMP / "val", 400, seed=2)
    print("composites written:", n_tr, n_va, "in", round(time.time() - t0), "s")
except Exception:
    print(traceback.format_exc()[-1800:])


Patches two crash bugs (random positions out of bounds in `synth.py`, and the occlusion step in `composite.py`), then re-lints and re-runs tests: 25 pass.

In [ ]:
sp = REPO / "src/anpr/ocr/synth.py"
s = sp.read_text()
s = s.replace("rng.randint(0, w - 40), rng.randint(0, h - 30)", "rng.randint(0, max(0, w - 40)), rng.randint(0, max(0, h - 30))")
sp.write_text(s)
cp = REPO / "src/anpr/data/composite.py"
c = cp.read_text()
c = c.replace("    patch = cv2.resize(plate, (tw, th), interpolation=cv2.INTER_AREA)\n    if rng.random() < 0.1:\n        patch = _occlude(patch, rng)\n", "    if rng.random() < 0.1 and min(plate.shape[:2]) >= 30:\n        plate = _occlude(plate, rng)\n    patch = cv2.resize(plate, (tw, th), interpolation=cv2.INTER_AREA)\n")
cp.write_text(c)
print("patched:", "max(0, w - 40)" in s, "plate = _occlude" in c)
print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -5; PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -3"))


Shows composite samples and measures plate size: the median plate covers only 0.34% of the image and 92% are under 2%, which is too small to be a useful training mix.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(14, 8))
rs = random.Random(3)
files = sorted((COMP / "train/images").glob("*.jpg"))
shown = [f for f in rs.sample(files, 60) if (COMP / "train/labels" / (f.stem + ".txt")).read_text().strip()][:12]
areas = []
for ax, f in zip(axes.ravel(), shown):
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for ln in (COMP / "train/labels" / (f.stem + ".txt")).read_text().splitlines():
        _, cx, cy, bw, bh = map(float, ln.split())
        cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)), (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (255, 0, 0), 2)
    ax.imshow(im)
    ax.axis("off")
plt.tight_layout()
plt.show()
al = np.array([float(l.split()[3]) * float(l.split()[4]) for f in (COMP / "train/labels").glob("*.txt") for l in f.read_text().splitlines()])
print("composite plates:", len(al), "| median area %.2f%%" % (100 * np.median(al)), "| share <2%%: %.0f%%" % (100 * (al < 0.02).mean()), "| empty images:", sum(1 for f in (COMP / "train/labels").glob("*.txt") if not f.read_text().strip()))


**Second attempt at the composite data.** Replaces `zoom_crop` and the composite logic so each image can be zoomed in on one vehicle. One test failed in this run; the median plate area rose to 0.80%.

In [ ]:
cp = REPO / "src/anpr/data/composite.py"
src = cp.read_text()
a, b = src.index("def make_composite("), src.index("def build_split(")
new_block = '''def zoom_crop(img: np.ndarray, vehicles: list[tuple[int, list[float]]], rng: random.Random):
    """Crop around one vehicle so it fills 30-95% of the width, like a close-up photo."""
    _, (x1, y1, x2, y2) = rng.choice(vehicles)
    cw = (x2 - x1) / rng.uniform(0.3, 0.95)
    ch = cw / rng.uniform(1.2, 1.8)
    cx = (x1 + x2) / 2 + rng.uniform(-0.1, 0.1) * cw
    cy = (y1 + y2) / 2 + rng.uniform(-0.1, 0.1) * ch
    H, W = img.shape[:2]
    nx1, ny1 = int(max(0, cx - cw / 2)), int(max(0, cy - ch / 2))
    nx2, ny2 = int(min(W, cx + cw / 2)), int(min(H, cy + ch / 2))
    if nx2 - nx1 < 64 or ny2 - ny1 < 48:
        return img, vehicles
    crop = img[ny1:ny2, nx1:nx2]
    scale = max(1.0, 480 / crop.shape[1])
    if scale > 1.0:
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    kept = []
    for cat, (bx1, by1, bx2, by2) in vehicles:
        qx1, qy1, qx2, qy2 = max(bx1, nx1), max(by1, ny1), min(bx2, nx2), min(by2, ny2)
        if (qx2 - qx1) >= 0.6 * (bx2 - bx1) and (qy2 - qy1) >= 0.6 * (by2 - by1):
            kept.append((cat, [(qx1 - nx1) * scale, (qy1 - ny1) * scale, (qx2 - nx1) * scale, (qy2 - ny1) * scale]))
    return crop, kept


def make_composite(
    img: np.ndarray,
    vehicles: list[tuple[int, list[float]]],
    pool: PlatePool,
    rng: random.Random,
    p_negative: float = 0.12,
    p_zoom: float = 0.7,
):
    """Return a (possibly zoomed) copy of img with plates pasted on up to three vehicles, plus boxes."""
    if vehicles and rng.random() < p_zoom:
        img, vehicles = zoom_crop(img, vehicles, rng)
    out = img.copy()
    boxes: list[list[float]] = []
    if not vehicles or rng.random() < p_negative:
        return out, boxes
    chosen = rng.sample(vehicles, k=min(len(vehicles), rng.choice([1, 1, 2, 3])))
    for cat, vb in chosen:
        plate = pool.sample(rng, two_line=True if cat == 4 and rng.random() < 0.5 else None)
        box = paste_plate(out, vb, cat, plate, rng)
        if box is not None:
            boxes.append(box)
    if boxes and rng.random() < 0.5:
        ok, enc = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, rng.randint(40, 92)])
        out = cv2.imdecode(enc, cv2.IMREAD_COLOR) if ok else out
    return out, boxes


'''
cp.write_text(src[:a] + new_block + src[b:])
print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -6; PYTHONPATH=src python -m pytest -q tests/test_composite.py 2>&1 | tail -4"))
import importlib
import shutil

import anpr.data.composite as _c

importlib.reload(_c)
from anpr.data.composite import build_split

shutil.rmtree(COMP, ignore_errors=True)
n_tr = build_split(CO / "val2017", tr_names, veh, pool, COMP / "train", 3000, seed=1)
n_va = build_split(CO / "val2017", va_names, veh, pool, COMP / "val", 400, seed=2)
al = np.array([float(l.split()[3]) * float(l.split()[4]) for f in (COMP / "train/labels").glob("*.txt") for l in f.read_text().splitlines()])
print("composites:", n_tr, n_va, "| plates:", len(al), "| median area %.2f%%" % (100 * np.median(al)), "| share <2%%: %.0f%%" % (100 * (al < 0.02).mean()), "| share <0.5%%: %.0f%%" % (100 * (al < 0.005).mean()))


Rewrites the zoom to be log-uniform (vehicle spans 30% to 220% of the crop width, from wide shot to close-up). Tests pass; median plate area is 0.90% and 73% of plates are still under 2%.

In [ ]:
cp = REPO / "src/anpr/data/composite.py"
src = cp.read_text()
a, b = src.index("def zoom_crop("), src.index("def make_composite(")
new_zoom = '''def zoom_crop(img: np.ndarray, vehicles: list[tuple[int, list[float]]], rng: random.Random):
    """Crop around one vehicle; it spans 30% to 220% of the crop width (wide shot to close-up)."""
    _, (x1, y1, x2, y2) = rng.choice(vehicles)
    frac = float(np.exp(rng.uniform(np.log(0.3), np.log(2.2))))
    cw = (x2 - x1) / frac
    ch = cw / rng.uniform(1.2, 1.8)
    cx = (x1 + x2) / 2 + rng.uniform(-0.12, 0.12) * (x2 - x1)
    cy = (y1 + y2) / 2 if frac <= 1 else y1 + 0.75 * (y2 - y1)
    cy += rng.uniform(-0.1, 0.1) * ch
    H, W = img.shape[:2]
    nx1, ny1 = int(max(0, cx - cw / 2)), int(max(0, cy - ch / 2))
    nx2, ny2 = int(min(W, cx + cw / 2)), int(min(H, cy + ch / 2))
    if nx2 - nx1 < 48 or ny2 - ny1 < 36:
        return img, vehicles
    crop = img[ny1:ny2, nx1:nx2]
    scale = max(1.0, 480 / crop.shape[1])
    if scale > 1.0:
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    kept = []
    for cat, (bx1, by1, bx2, by2) in vehicles:
        if nx1 <= (bx1 + bx2) / 2 <= nx2 and ny1 <= (by1 + by2) / 2 <= ny2:
            kept.append((cat, [(bx1 - nx1) * scale, (by1 - ny1) * scale, (bx2 - nx1) * scale, (by2 - ny1) * scale]))
    return crop, kept


'''
cp.write_text(src[:a] + new_zoom + src[b:])
tp = REPO / "tests/test_composite.py"
t = tp.read_text()
t = t.replace("make_composite(img, vehicles, pool, rng, p_negative=0.0)", "make_composite(img, vehicles, pool, rng, p_negative=0.0, p_zoom=0.0)")
t += '''

def test_zoomed_composite_boxes_stay_inside_output():
    rng = random.Random(3)
    img = np.full((480, 640, 3), 90, np.uint8)
    vehicles = [(3, [100.0, 100.0, 500.0, 420.0])]
    pool = PlatePool([], synth_frac=1.0)
    for _ in range(30):
        out, boxes = make_composite(img, vehicles, pool, rng, p_negative=0.0, p_zoom=1.0)
        h, w = out.shape[:2]
        assert all(0 <= b[0] < b[2] <= w and 0 <= b[1] < b[3] <= h for b in boxes)
'''
tp.write_text(t)
print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -6; PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -4"))
importlib.reload(_c)
from anpr.data.composite import build_split

shutil.rmtree(COMP, ignore_errors=True)
n_tr = build_split(CO / "val2017", tr_names, veh, pool, COMP / "train", 3000, seed=1)
n_va = build_split(CO / "val2017", va_names, veh, pool, COMP / "val", 400, seed=2)
al = np.array([float(l.split()[3]) * float(l.split()[4]) for f in (COMP / "train/labels").glob("*.txt") for l in f.read_text().splitlines()])
print("composites:", n_tr, n_va, "| plates:", len(al), "| median area %.2f%%" % (100 * np.median(al)), "| share <2%%: %.0f%%" % (100 * (al < 0.02).mean()), "| share <0.5%%: %.0f%%" % (100 * (al < 0.005).mean()), "| share >10%%: %.0f%%" % (100 * (al >= 0.10).mean()))


Tunes the zoom further (50% to 300% of crop width, zoom probability 0.9). The composites now have a median plate area of 1.75%, with 7% of plates above 10% of the image. Archives the dataset (253 MB) to Drive.

In [ ]:
src = cp.read_text()
src = src.replace("np.log(0.3), np.log(2.2)", "np.log(0.5), np.log(3.0)").replace("p_zoom: float = 0.7", "p_zoom: float = 0.9").replace("spans 30% to 220%", "spans 50% to 300%")
cp.write_text(src)
print(sh("cd /content/ANPR && ruff format src tests; ruff check --fix src tests | tail -4; PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -2"))
importlib.reload(_c)
from anpr.data.composite import build_split

shutil.rmtree(COMP, ignore_errors=True)
n_tr = build_split(CO / "val2017", tr_names, veh, pool, COMP / "train", 3200, seed=1)
n_va = build_split(CO / "val2017", va_names, veh, pool, COMP / "val", 400, seed=2)
al = np.array([float(l.split()[3]) * float(l.split()[4]) for f in (COMP / "train/labels").glob("*.txt") for l in f.read_text().splitlines()])
print("composites:", n_tr, n_va, "| plates:", len(al), "| median area %.2f%%" % (100 * np.median(al)), "| share <2%%: %.0f%%" % (100 * (al < 0.02).mean()), "| share <0.5%%: %.0f%%" % (100 * (al < 0.005).mean()), "| share >10%%: %.0f%%" % (100 * (al >= 0.10).mean()))
shutil.make_archive("/content/composite_v1", "tar", COMP)
shutil.copy2("/content/composite_v1.tar", D / "data/processed/composite_v1.tar")
print("archived to Drive:", round((D / "data/processed/composite_v1.tar").stat().st_size / 1e6), "MB")


Prepares the final detector training sets: loads the Saisirishan and Kaggle splits (test 1,006 and validation 678 plates), removes duplicates, and builds `det_main` (4,743 training images) and `det_kaggle` (1,543 training images), both validated on the same 678 images.

In [ ]:
import json
import os
import pickle
import traceback
import xml.etree.ElementTree as ET

import pandas as pd

from anpr.ocr.consensus import normalize_plate

try:
    split_tab = pd.read_csv(D / "data/external/ocr_eval_v1/saisirishan_split.csv", keep_default_na=False)
    text_split = dict(zip(split_tab.gtx, split_tab.split))
    sai_boxes = pd.read_csv(D / "data/external/det_eval_v1/saisirishan_boxes.csv", keep_default_na=False)
    sai_boxes["boxes"] = sai_boxes.boxes.map(json.loads)
    SSD = Path("/content/ext/saisirishan")
    name2path = {p.name: p for p in SSD.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    # plate text per image from the VOC XML (object name containing a digit)
    img_text = {}
    for p in SSD.rglob("*.xml"):
        root = ET.parse(p).getroot()
        for obj in root.iter("object"):
            nm = obj.findtext("name") or ""
            if any(ch.isdigit() for ch in nm):
                stem = root.findtext("filename") or p.name[:-4]
                img_text[Path(stem).name] = normalize_plate(nm)
                img_text[p.name[:-4]] = normalize_plate(nm)
    rows = []
    for r in sai_boxes.itertuples():
        txt = img_text.get(r.image) or img_text.get(Path(r.image).stem)
        sp = text_split.get(txt)
        path = name2path.get(r.image)
        if sp and path:
            rows.append({"image": str(path), "w": r.w, "h": r.h, "boxes": r.boxes, "split": sp, "folder": r.folder})
    sai = pd.DataFrame(rows)
    print("saisirishan images with split:", sai.split.value_counts().to_dict(), "of", len(sai_boxes), flush=True)
    dc = pd.read_csv(D / "data/external/det_eval_v1/datacluster_boxes.csv", keep_default_na=False)
    dc["boxes"] = dc.boxes.map(json.loads)
    dc["image"] = [str(next(Path("/content/ext/datacluster").rglob(n))) for n in dc.image]

    def yolo_lines(boxes, w, h):
        return [("0 %.6f %.6f %.6f %.6f" % ((b[0] + b[2]) / 2 / w, (b[1] + b[3]) / 2 / h, (b[2] - b[0]) / w, (b[3] - b[1]) / h)) for b in boxes]

    def link_into(dst, items):
        (dst / "images").mkdir(parents=True, exist_ok=True)
        (dst / "labels").mkdir(parents=True, exist_ok=True)
        for src_img, lab_lines, tag in items:
            ext = Path(src_img).suffix
            stem = tag
            os.symlink(src_img, dst / "images" / (stem + ext))
            (dst / "labels" / (stem + ".txt")).write_text(chr(10).join(lab_lines) + (chr(10) if lab_lines else ""))

    def kaggle_items(split):
        out = []
        for lf in sorted((Y / split / "labels").glob("*.txt")):
            img = next(p for p in [Y / split / "images" / (lf.stem + e) for e in (".png", ".jpg")] if p.exists())
            out.append((str(img), [ln for ln in lf.read_text().splitlines() if ln.strip()], "k_" + lf.stem))
        return out

    def comp_items(split):
        out = []
        for lf in sorted((COMP / split / "labels").glob("*.txt")):
            out.append((str(COMP / split / "images" / (lf.stem + ".jpg")), [ln for ln in lf.read_text().splitlines() if ln.strip()], "c_" + lf.stem))
        return out

    val_items = [(r.image, yolo_lines(r.boxes, r.w, r.h), "s_" + str(i)) for i, r in enumerate(sai[sai.split == "val"].itertuples())]
    for name, train in (("det_main", kaggle_items("train") + comp_items("train")), ("det_kaggle", kaggle_items("train"))):
        root = Path("/content") / name
        shutil.rmtree(root, ignore_errors=True)
        link_into(root / "train", train)
        link_into(root / "valid", val_items)
        link_into(root / "test", kaggle_items("test"))
        (root / "data.yaml").write_text("path: " + str(root) + chr(10) + "train: train/images" + chr(10) + "val: valid/images" + chr(10) + "test: test/images" + chr(10) + "names:" + chr(10) + "  0: license_plate" + chr(10))
        print(name, "train images:", len(train), "| valid:", len(val_items))

    def eval_entry(images, boxes_list):
        import cv2

        areas, paths = [], []
        for p in images:
            h, w = cv2.imread(p).shape[:2]
            areas.append(float(w * h))
            paths.append(p)
        return {"images": paths, "gts": [np.array(b, dtype=float).reshape(-1, 4) for b in boxes_list], "areas": areas}

    def kaggle_eval(split):
        paths, gts = [], []
        for src_img, lines, _ in kaggle_items(split):
            im = cv2.imread(src_img)
            h, w = im.shape[:2]
            bx = []
            for ln in lines:
                _, cx, cy, bw, bh = map(float, ln.split()[:5])
                bx.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            paths.append(src_img)
            gts.append(bx)
        return eval_entry(paths, gts)

    sv, stt = sai[sai.split == "val"], sai[sai.split == "test"]
    eval_sets = {
        "saisirishan_val": eval_entry(list(sv.image), list(sv.boxes)),
        "saisirishan_test": eval_entry(list(stt.image), list(stt.boxes)),
        "datacluster": eval_entry(list(dc.image), list(dc.boxes)),
        "kaggle_test": kaggle_eval("test"),
    }
    print({k: (len(v["images"]), int(sum(len(g) for g in v["gts"]))) for k, v in eval_sets.items()})
    pickle.dump(eval_sets, open("/content/eval_sets.pkl", "wb"))
    shutil.copy2("/content/eval_sets.pkl", D / "data/external/det_eval_v1/eval_sets_paths.pkl")
except Exception:
    print(traceback.format_exc()[-2000:])
